# Table 2: Pooled Model Selection

## Step 1: Pooled multi-model comparison

This step compares Linear Regression, SVR, Random Forest, GBM, MLP, and XGBoost using 5-fold cross-validation on the pooled all-alloy dataset with the Raw-18 + alloy one-hot feature set.


In [1]:
"""
Pooled all-alloy creep rupture life prediction: multi-model comparison.

Feature set:
    Raw-18 + alloy one-hot = [stress, temperature, 16 composition features, alloy indicators]

Notes:
    - All alloys are mixed into one dataset.
    - Alloy one-hot columns are added so the model is not forced to learn all
      alloy-to-alloy offsets from composition alone.
    - Geometry variables are not used here; this isolates model selection from
      the proposed geometry-correction method.
"""

import os
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

FEATURES = [
    "Stress (Mpa)",
    "Temperature of Creep Test (C)",
    "C, wt.%",
    "Si, wt.%",
    "Mn, wt.%",
    "P, wt.%",
    "S, wt.%",
    "Ni, wt.%",
    "Cr, wt.%",
    "Mo, wt.%",
    "Al, wt.%",
    "N, wt.%",
    "Ti, wt.%",
    "Fe, wt.%",
    "Nb, wt.%",
    "B, wt.%",
    "Cu, wt.%",
    "Co, wt.%",
]

N_SPLITS = 5
GPR_MAX_N = 150

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}


def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """RPV Cross-Sectional Area """
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing = rpv_mask & df["Cross-Sectional Area"].isna()

    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[
        rpv_mask & df["Cross-Sectional Area"].notna(), "Cross-Sectional Area"
    ].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask
            & (df["Alloy Designation"] == alloy)
            & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area",
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(
        f"  [RPV] imputed Cross-Sectional Area: "
        f"{n_cyl} cylindrical (formula), {n_flat} flat (median)"
    )
    return df


def impute_ss316l_composition(df: pd.DataFrame) -> pd.DataFrame:
    """SS316L composition """
    df = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[FEATURES[2]].isna()
    if not comp_missing.any():
        return df

    medians = df.loc[mask & ~comp_missing, FEATURES[2:]].median()
    for col in FEATURES[2:]:
        df.loc[comp_missing, col] = medians[col]
    print(
        f"  [SS316L] imputed composition for {int(comp_missing.sum())} records "
        f"using SS316L per-column medians"
    )
    return df


def make_models(n):
    models = {
        "Linear Regression": LinearRegression(),
        "SVR": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("svr", SVR(kernel="rbf", C=10, epsilon=0.05, gamma="scale")),
            ]
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=500,
            max_depth=10,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        ),
        "GBM": GradientBoostingRegressor(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.8,
            min_samples_leaf=3,
            random_state=42,
        ),
        "MLP": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "mlp",
                    MLPRegressor(
                        hidden_layer_sizes=(128, 64, 32),
                        activation="relu",
                        max_iter=1000,
                        learning_rate_init=0.001,
                        random_state=42,
                        early_stopping=True,
                        validation_fraction=0.1,
                    ),
                ),
            ]
        ),
    }

    if XGBRegressor is not None:
        models["XGBoost"] = XGBRegressor(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            random_state=42,
            verbosity=0,
            n_jobs=1,
        )

    if n <= GPR_MAX_N:
        models["GPR"] = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "gpr",
                    GaussianProcessRegressor(
                        kernel=ConstantKernel(1.0) * RBF(length_scale=1.0)
                        + WhiteKernel(noise_level=0.1),
                        n_restarts_optimizer=3,
                        random_state=42,
                    ),
                ),
            ]
        )

    return models


def metric_dict(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "within_2x_pct": float(np.mean(np.abs(y_true - y_pred) < np.log10(2)) * 100.0),
        "within_3x_pct": float(np.mean(np.abs(y_true - y_pred) < np.log10(3)) * 100.0),
    }


def load_pooled_raw18():
    print("Loading data ...")
    df = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
    df = impute_rpv_cross_section(df)
    df = impute_ss316l_composition(df)

    mask = (
        df["Alloy Designation"].notna()
        & (df["Notch"] == "unnotched")
        & df["Creep Rupture Life (h)"].notna()
        & (df["Creep Rupture Life (h)"] > 0)
        & df["Stress (Mpa)"].notna()
        & (df["Stress (Mpa)"] > 0)
        & df["Specimen Shape"].isin(["Flat", "Cylindrical"])
    )
    data = df[mask].copy().reset_index(drop=True)
    data["log_tR"] = np.log10(data["Creep Rupture Life (h)"])

    required = FEATURES + ["Alloy Designation", "log_tR"]
    sub = data[required].dropna(subset=required).reset_index(drop=True)

    alloy_dummies = pd.get_dummies(sub["Alloy Designation"], prefix="alloy").astype(float)
    x = np.column_stack([sub[FEATURES].values.astype(float), alloy_dummies.values])
    y = sub["log_tR"].values.astype(float)

    print(f"  Samples: {len(y)}")
    print(f"  Alloys:  {alloy_dummies.shape[1]}")
    print(f"  Features: {x.shape[1]} = Raw-18 + {alloy_dummies.shape[1]} alloy indicators")
    return sub, x, y, alloy_dummies.shape[1]


def run_cv_models(x, y, n_alloys):
    print("\nRunning 5-fold CV ...")
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    models = make_models(len(y))
    rows = []

    for name, model in models.items():
        print(f"\n  [{name}]")
        t0 = time.time()
        y_pred = np.zeros(len(y))

        for fold, (train_idx, val_idx) in enumerate(kf.split(x), 1):
            fitted = clone(model)
            fitted.fit(x[train_idx], y[train_idx])
            y_pred[val_idx] = fitted.predict(x[val_idx])
            print(f"    Fold {fold}/{N_SPLITS} done", end="\r")

        elapsed = time.time() - t0
        m = metric_dict(y, y_pred)
        rows.append(
            {
                "analysis": "pooled_all_data_model_comparison",
                "feature_set": "Raw-18 + alloy one-hot",
                "model": name,
                "status": "ok",
                "n": len(y),
                "n_alloys": n_alloys,
                "n_features": x.shape[1],
                **m,
                "time_s": elapsed,
                "note": "",
            }
        )
        print(
            f"    R2={m['r2']:.4f}  RMSE={m['rmse']:.4f}  "
            f"±2x={m['within_2x_pct']:.1f}%  [{elapsed:.1f}s]"
        )

    if len(y) > GPR_MAX_N:
        rows.append(
            {
                "analysis": "pooled_all_data_model_comparison",
                "feature_set": "Raw-18 + alloy one-hot",
                "model": "GPR",
                "status": "skipped",
                "n": len(y),
                "n_alloys": n_alloys,
                "n_features": x.shape[1],
                "r2": np.nan,
                "rmse": np.nan,
                "within_2x_pct": np.nan,
                "within_3x_pct": np.nan,
                "time_s": np.nan,
                "note": f"Skipped because pooled n>{GPR_MAX_N}; GPR training is too expensive for this pooled comparison.",
            }
        )

    result_df = pd.DataFrame(rows).sort_values(
        ["status", "r2"], ascending=[True, False], na_position="last"
    )

    print(f"\n{'=' * 84}")
    print(f"{'Model':<18} {'Status':<8} {'R2':>8} {'RMSE':>8} {'±2x%':>8} {'±3x%':>8} {'Time(s)':>9}")
    print("-" * 84)
    for _, row in result_df.iterrows():
        r2 = "" if pd.isna(row["r2"]) else f"{row['r2']:.4f}"
        rmse = "" if pd.isna(row["rmse"]) else f"{row['rmse']:.4f}"
        w2 = "" if pd.isna(row["within_2x_pct"]) else f"{row['within_2x_pct']:.1f}"
        w3 = "" if pd.isna(row["within_3x_pct"]) else f"{row['within_3x_pct']:.1f}"
        ts = "" if pd.isna(row["time_s"]) else f"{row['time_s']:.1f}"
        print(f"{row['model']:<18} {row['status']:<8} {r2:>8} {rmse:>8} {w2:>8} {w3:>8} {ts:>9}")
    print("=" * 84)

    return result_df


def run_table2_model_comparison():
    _, x, y, n_alloys = load_pooled_raw18()
    return run_cv_models(x, y, n_alloys)


In [2]:
# Run Step 1
results_table2 = run_table2_model_comparison()


Loading data ...
  [RPV] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L] imputed composition for 15 records using SS316L per-column medians
  Samples: 2433
  Alloys:  12
  Features: 30 = Raw-18 + 12 alloy indicators

Running 5-fold CV ...

  [Linear Regression]
    R2=0.5616  RMSE=0.7657  ±2x=30.7%  [0.0s]

  [SVR]
    R2=0.8702  RMSE=0.4167  ±2x=71.2%  [1.7s]

  [Random Forest]
    R2=0.8437  RMSE=0.4571  ±2x=59.4%  [4.9s]

  [GBM]
    R2=0.9040  RMSE=0.3582  ±2x=74.1%  [10.3s]

  [MLP]
    R2=0.9015  RMSE=0.3630  ±2x=77.0%  [7.7s]

  [XGBoost]
    R2=0.9009  RMSE=0.3640  ±2x=72.9%  [1.9s]

Model              Status         R2     RMSE     ±2x%     ±3x%   Time(s)
------------------------------------------------------------------------------------
GBM                ok         0.9040   0.3582     74.1     88.6      10.3
MLP                ok         0.9015   0.3630     77.0     89.4       7.7
XGBoost            ok         0.9009   0.3640     72.9    

## Final Table 2


In [3]:
MODEL_NAME_MAP = {
    "GBM": "Gradient boosting machine",
    "XGBoost": "Extreme gradient boosting",
    "MLP": "Multilayer perceptron",
    "SVR": "Support vector regression",
    "Random Forest": "Random forest",
    "Linear Regression": "Linear regression",
}
MODEL_ORDER = ["GBM", "XGBoost", "MLP", "SVR", "Random Forest", "Linear Regression"]
COLUMN_MAP = {
    "ML Model": "ML Model",
    "r2": "R2",
    "rmse": "RMSE",
    "within_2x_pct": "Within 2x (%)",
    "time_s": "Time (s)",
    "n": "n",
}

paper = results_table2[
    (results_table2["status"] == "ok") & results_table2["model"].isin(MODEL_ORDER)
].copy()
paper["ML Model"] = paper["model"].map(MODEL_NAME_MAP)
paper["rank"] = paper["model"].map({model: i for i, model in enumerate(MODEL_ORDER)})
paper = paper.sort_values("rank")
final_table = paper[list(COLUMN_MAP)].rename(columns=COLUMN_MAP).reset_index(drop=True)
final_table


,ML Model,R2,RMSE,Within 2x (%),Time (s),n
0,Gradient boosting machine,0.904042,0.358233,74.106042,10.300038,2433
1,Extreme gradient boosting,0.900908,0.364036,72.872996,1.941864,2433
2,Multilayer perceptron,0.901489,0.362968,76.983148,7.655532,2433
3,Support vector regression,0.870186,0.416663,71.228935,1.655595,2433
4,Random forest,0.843748,0.457128,59.350596,4.862011,2433
5,Linear regression,0.561560,0.765738,30.661734,0.020946,2433
